In [ ]:
import os, json
os.environ["AF3_NB_OVERRIDES"] = json.dumps({
    "model": "openbind0",
    "protein": "SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQ",
    "msa_mode": "single_sequence",
    "num_diffusion_samples": 1,
    "num_recycles": 3,
    "jobname": "vops",
})
print("overrides set")


overrides set


In [ ]:
#@title Install dependencies (~35 s)
import os, time, glob, shutil, sys, subprocess
_T0 = time.time()

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model AND its lowering in
#@markdown   Drive so the next session skips both. Measured on a Colab T4 (58
#@markdown   residues): 72 s for the first fold, 26 s once the cache is there. The
#@markdown   cache populates itself -- nothing is downloaded -- so the saving starts
#@markdown   with your second fold. Never changes a result.

# Set any form field from the environment, for runs outside Colab:
#   AF3_NB_OVERRIDES='{"model": "boltz2"}'
import json as _json
for _k, _v in _json.loads(os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

VERSION = '3.1.11'          # the wheel, from PyPI
# Where the PYTHON comes from. A tag (`v3.1.11`) is the shipping notebook; the
# `colab` branch carries the experimental live-animation and steering code,
# which no release has. See the overlay below.
SOURCE = 'colab'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
AF2_DIR = 'af2_params'
IS_AF3 = (model == 'alphafold3')
IS_AF2 = model.startswith('af2_')
# int8 weights, expanded on load; AF2 and AF3 ship their own float32 files.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'


def _sh(cmd, what):
  """Run a shell command, raising if it fails."""
  if os.system(cmd) != 0:
    raise RuntimeError(f'{what} failed. The output is above.')


if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # Installed with --no-deps, so the package's own imports are listed here.
  # Letting pip resolve them would re-download jax and the CUDA stack.
  # tokamax 0.0.11 WORKS ON jax 0.11.1 AND BREAKS ON 0.11.2, where
  # `jax.experimental.hijax.HiPrimitive` is gone -- and with tokamax
  # unimportable, nothing in the stack loads. A GPU image already has 0.11.1,
  # satisfies tokamax's `jax>=0.9.1`, and pip leaves it alone; a TPU image has
  # 0.7.2, FAILS that requirement, and pip upgrades it to the latest.
  # So LOOK BEFORE INSTALLING. Re-pinning a jax that is already right costs
  # 38 s and a libtpu download for nothing, and on a GPU image touching jax at
  # all would drag in the CUDA stack -- which is the very thing --no-deps is
  # here to avoid.
  import glob as _glob
  import importlib.metadata as _md
  try:
    _jax_now = _md.version('jax')
  except Exception:
    _jax_now = None
  _is_tpu = bool(_glob.glob('/dev/accel*') or os.environ.get('TPU_ACCELERATOR_TYPE'))
  if _jax_now == '0.11.1':
    pass                     # every GPU image today: nothing to do
  elif _is_tpu:
    print(f'jax {_jax_now} on a TPU runtime; pinning to 0.11.1 for tokamax')
    _sh('pip install -q "jax[tpu]==0.11.1"', 'pinning jax for the TPU runtime')
  else:
    # Deliberately NOT fixed here: jax[cuda12] would re-download the CUDA
    # stack. Say it plainly instead -- if the Colab GPU image ever moves to
    # 0.11.2, this line is what explains the import errors that follow.
    print(f'NOTE: jax {_jax_now} is not the 0.11.1 tokamax 0.0.11 expects; '
          'if imports fail with hijax.HiPrimitive, pin jax to 0.11.1.')
  # A PRE-AMPERE CARD HAS NO OTHER FUSED ATTENTION. cuDNN's SDPA wants SM80,
  # tokamax has no kernel for it, and XLA gates Pallas/Triton at sm_80 -- so a
  # T4, the commonest Colab GPU, runs the materialising XLA path for the 77%
  # of a pairformer pass that is triangle attention. Milot Mirdita's
  # colabfold-legacy-kernels is the exception: 3.0-3.35x that path on a T4,
  # measured at this model's own shape. Linux x86_64 wheels, and only where
  # the card can use them.
  import platform as _pyplat
  _cc0 = None
  if _pyplat.system() == 'Linux' and _pyplat.machine() == 'x86_64':
    try:
      _out0 = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap',
                              '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=15).stdout
      _cc0 = min(float(x) for x in _out0.split() if x.strip())
    except Exception:
      _cc0 = None
  if _cc0 is not None and _cc0 < 8.0:
    _sh('pip install -q colabfold-legacy-kernels==0.2.0',
        'installing the pre-Ampere fused kernels')
  # zstandard IS one of them -- params.py, post_processing.py and
  # folding_input.py all import it. It happened to be preinstalled on the
  # GPU images, so its absence only showed up on a TPU runtime, as
  # `ModuleNotFoundError: No module named zstandard` from inside the fold,
  # long after the install cell had reported success.
  _sh("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 "
      "tokamax==0.0.11 ml_collections zstandard", 'installing dependencies')
  _sh("pip install -q git+https://github.com/sokrypton/py2Dmol.git",  # wheel lags the repo
      'installing py2Dmol')
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")  # AF2's tar is 5.3 GB
  # Retried: PyPI's index can lag a just-published release by a few minutes.
  for _try in range(4):
    if os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}') == 0:
      break
    print(f'pip could not find {VERSION} yet; retrying in 20 s')
    time.sleep(20)
  else:
    raise RuntimeError(f'could not install alphafold3-colabfold=={VERSION}')

  # haiku 0.0.17 still calls the moved jax.core.DropVar.
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  import alphafold3  # confirms the install before anything depends on it
  os.system('touch ALPHAFOLD3_READY')
  print(f'Packages installed ({alphafold3.__file__}).')

# THE BRANCH OVERLAY RUNS EVERY TIME, not only on a fresh install. It used to
# sit inside the `ALPHAFOLD3_READY` guard, so a session that had already
# installed never refreshed it -- and a file ADDED on the branch (staged.py)
# could never arrive at all: `ImportError: cannot import name staged`, on a
# notebook whose overlay list had already been fixed. Five wgets, so there is
# no reason to skip them.
# EXPERIMENTAL BRANCH OVERLAY. The live cell needs library changes that are
# not in any release, so the branch's Python is copied over the installed
# wheel. That is legitimate here and nowhere else: `colab` changes no C++, so
# the compiled extension in the wheel is exactly the one this source expects.
# The moment a .cc changes on the branch this stops being true, which is what
# the assertion below is for -- and then the install becomes
#     pip install git+https://github.com/sokrypton/alphafold3@colab
# which builds the extension from scratch (~10 minutes on Colab).
# run_alphafold.py is a top-level script, not part of the package, and it is
# fetched HERE rather than under the install guard for the same reason as the
# rest of the overlay: on a warm session the guard is skipped and a stale copy
# would survive a push.
# run_alphafold.py COMES OUT OF THE SAME TARBALL as the package, below.
#
# It used to be its own wget from raw.githubusercontent, and that host serves a
# stale blob for a long time after a push -- a cache-buster query string did
# NOT help (verified: a fresh wget on the VM returned a file missing the
# newest commit while the GitHub API reported that commit as the branch head).
# Two failures came of it, and both looked like code bugs: the fold died on a
# validation the branch no longer had, quoting an error message that no longer
# existed. It is also the same shape as the overlay-list bug -- the script and
# the package coming from different places and disagreeing. One snapshot, one
# fetch, no way for them to drift.

# ONE TARBALL, for the script and (on a branch) the package. A tag needs
# run_alphafold.py too -- it is not part of the wheel -- so this runs either
# way and only the package overlay is conditional.
import importlib.metadata as _md
_root = os.path.dirname(_md.distribution('alphafold3-colabfold')
                        .locate_file('alphafold3'))
_ref = ('refs/heads/' + SOURCE if SOURCE != f'v{VERSION}'
        else 'refs/tags/' + SOURCE)
_sh(f'wget -q -O branch.tar.gz https://codeload.github.com'
    f'/sokrypton/alphafold3/tar.gz/{_ref}?nocache={int(time.time())}',
    f'fetching {SOURCE}')
_sh('rm -rf branch_src && mkdir branch_src && '
    'tar xzf branch.tar.gz -C branch_src --strip-components=1', 'unpacking it')
_sh('cp branch_src/run_alphafold.py run_alphafold.py',
    'taking run_alphafold.py from the tarball')

if SOURCE != f'v{VERSION}':
  # THE WHOLE TREE, not a list of files. There WAS a list, fetched from the
  # branch so it could not go stale against the branch -- and it went stale
  # against the WHEEL instead, which is the comparison that actually matters:
  # `git diff main colab` is empty for a file that main changed AFTER the
  # release was cut, so evoformer.py stayed at 3.1.11 while run_alphafold.py
  # came from the branch and asked it for bfloat16='intermediate'. The wheel's
  # assert had never heard of that value and every fold died in the first
  # recycle. A tarball of the branch has no list to keep in step: 8 MB, about
  # a second, one request instead of six.
  # `.` copies the CONTENTS, merging into the installed package, so the
  # compiled extension (.so) stays where the wheel put it -- which is the
  # whole reason overlaying source on a wheel is legitimate here.
  _sh(f'cp -r branch_src/src/alphafold3/. {_root}/alphafold3/',
      'overlaying the package')
  _sh('cp branch_src/dev/live/live_frames.py live_frames.py',
      'fetching live_frames.py')
  print(f'overlaid the {SOURCE} branch onto the {VERSION} wheel')

if SOURCE != f'v{VERSION}':
  # Proves the overlay landed rather than trusting the download. Checked by
  # READING the files: importing run_alphafold here would pull in
  # alphafold3.constants, whose pickles the input cell has not written yet
  # (FileNotFoundError: chemical_component_sets.pickle), and the failure
  # would land before CACHE_DIR is even defined.
  import importlib.metadata as _md2
  _r = os.path.dirname(_md2.distribution('alphafold3-colabfold')
                       .locate_file('alphafold3'))
  for _f, _needle in (
      ('run_alphafold.py', 'def live_model'),
      (os.path.join(_r, 'alphafold3/model/model.py'), "stage='all'"),
      (os.path.join(_r, 'alphafold3/model/staged.py'), 'def make_stages'),
  ):
    assert _needle in open(_f).read(), (
        f'{_f} is not the {SOURCE} version ({_needle!r} missing) -- the '
        'overlay did not take effect and the live cell would fail')

# tokamax's Triton kernels need more shared memory than Ada cards have, so
# restrict them to datacenter GPUs (A100 cc 8.0, H100 cc 9.0+).
#
# DO NOT IMPORT tokamax TO DO IT, AND DO NOT DO IT OFF A GPU. `import tokamax`
# imports jax, and on a TPU runtime the first process to touch jax OWNS the
# chip -- so this line, whose only job is to edit a CUDA policy, took the TPU
# and left the fold's own subprocess with
#     ABORTED: The TPU is already in use by process with pid <the kernel>
# and a silent fall back to "CPU-only inference". find_spec locates the file
# without executing the package, and the patch is skipped where it means
# nothing. (platform.detect_device() is jax-free by design -- checked.)
from alphafold3.model.components import platform as _plat
_dev0, _cap0 = _plat.detect_device()
if _dev0 != 'gpu' or not _plat.needs_tokamax_patch(_cap0):
  print(f'tokamax patch not needed on this device ({_dev0}, cc {_cap0}).')
try:
  import importlib.util as _ilu
  if _dev0 != 'gpu' or not _plat.needs_tokamax_patch(_cap0):
    raise SystemExit  # caught below; nothing to patch
  _spec = _ilu.find_spec('tokamax')
  _gu = os.path.join(os.path.dirname(_spec.origin), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except SystemExit:
  pass
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, fetched in the background by the same code the run uses.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if IS_AF3 and not os.path.isfile(STAMP):
  # DeepMind's own release, subject to the AlphaFold 3 terms of use, which
  # run_alphafold prints at startup. A copy you already have in NATIVE_DIR is
  # used as-is.
  os.makedirs(NATIVE_DIR, exist_ok=True)
  if glob.glob(f'{NATIVE_DIR}/*.bin.zst'):
    open(STAMP, 'w').close()
    print(f'Using the AlphaFold 3 parameters already in {NATIVE_DIR}/.')
  else:
    print('Downloading AlphaFold 3 parameters (~1 GB)...')
    os.system(f'(wget -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"'
              f' > {STAMP}.log 2>&1 && touch {STAMP}) &')
elif not (IS_AF3 or os.path.isfile(STAMP)):
  _script, _args = ('prefetch_af2.py', AF2_DIR) if IS_AF2 else (
      'prefetch_weights.py', f'{model} {PRECISION}')
  print(f'Downloading {"official AlphaFold 2 parameters (CC BY 4.0)" if IS_AF2 else model} weights...')
  with open(_script, 'w') as fh:
    fh.write('import sys\n'
             'from alphafold3.model import weights\n'
             + ('print(weights.ensure_af2_params(sys.argv[1]))\n' if IS_AF2 else
                'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n'))
  os.system(f'(python {_script} {_args} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# /tmp is wiped with the VM, so a fresh session recompiles (~53 s); Drive survives.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')


def _await(sentinel, limit=1200):
  """Wait for a background job, reporting its log if it never finishes."""
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} \u2713  ({time.time() - t0:.0f} s)')


_await(STAMP)

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('the AlphaFold 3 download is incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')
print(f'Setup took {time.time() - _T0:.0f} s.')


override: model = 'openbind0'
Installing packages...


Packages installed (/usr/local/lib/python3.13/dist-packages/alphafold3/__init__.py).


overlaid the colab branch onto the 3.1.11 wheel
tokamax patch not needed on this device (gpu, cc 7.5).


WEIGHTS_DONE_openbind0_int8 ✓  (10 s)
Setup complete!  Model: openbind0.
Setup took 37 s.


In [ ]:
# 1. THE ORACLE, on the card itself: layout (device-independent) + kernel numerics.
import subprocess, sys
p = subprocess.run([sys.executable, 'branch_src/dev/oracles/volta_ops_check.py'],
                   capture_output=True, text=True)
print(p.stdout)
print(p.stderr[-3000:] if p.returncode else '')
print('oracle exit', p.returncode)



Traceback (most recent call last):
  File "/content/branch_src/dev/oracles/volta_ops_check.py", line 30, in <module>
    from alphafold3.model.network import modules  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/alphafold3/model/network/modules.py", line 29, in <module>
    from . import diffusion_transformer
  File "/usr/local/lib/python3.13/dist-packages/alphafold3/model/network/diffusion_transformer.py", line 26, in <module>
    from alphafold3.model.atom_layout import atom_layout
  File "/usr/local/lib/python3.13/dist-packages/alphafold3/model/atom_layout/atom_layout.py", line 30, in <module>
    from alphafold3.constants import chemical_component_sets
  File "/usr/local/lib/python3.13/dist-packages/alphafold3/constants/chemical_component_sets.py", line 31, in <module>
    with open(_CCD_SETS_CCD_PICKLE_FILE, 'rb') as f:
         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or direc

In [ ]:
# 2. TriangleMultiplication, xla vs volta, at the trunk's own shape.
import time, jax, jax.numpy as jnp, numpy as np, haiku as hk
from alphafold3.model import model_config
from alphafold3.model.network import modules

def build(impl, n, c=128):
  gc = model_config.GlobalConfig(flash_attention_implementation=impl,
                                 bfloat16='all')
  cfg = modules.TriangleMultiplication.Config(equation='ikc,jkc->ijc')
  fwd = hk.transform(
      lambda a, m: modules.TriangleMultiplication(cfg, gc, name='tm')(a, m))
  act = (jax.random.normal(jax.random.PRNGKey(1), (n, n, c)) * 0.5).astype(jnp.bfloat16)
  mask = jnp.ones((n, n), jnp.bfloat16)
  params = fwd.init(jax.random.PRNGKey(0), act, mask)
  params = jax.tree.map(
      lambda v: (jax.random.normal(jax.random.PRNGKey(abs(hash(v.shape)) % 9999),
                                   v.shape) * 0.3).astype(v.dtype)
      if v.ndim > 1 else v, params)
  fn = jax.jit(lambda p, a, m: fwd.apply(p, None, a, m))
  return fn, params, act, mask

def timeit(fn, *a, iters=20):
  fn(*a).block_until_ready()
  t = time.time()
  for _ in range(iters):
    o = fn(*a)
  o.block_until_ready()
  return (time.time() - t) / iters * 1e3, np.asarray(o, np.float32)

for n in (256, 384):
  fx, px, ax, mx = build('xla', n)
  fv, pv, av, mv = build('volta', n)
  tx, ox = timeit(fx, px, ax, mx)
  tv, ov = timeit(fv, pv, av, mv)
  d = np.abs(ox - ov).max()
  print(f'N={n:4d}  xla {tx:7.3f} ms   volta {tv:7.3f} ms   {tx/tv:.2f}x   '
        f'max|d| {d:.4f} on |out| <= {np.abs(ox).max():.2f}')


FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/lib/python3.13/dist-packages/alphafold3/constants/converters/chemical_component_sets.pickle'

In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Fetch chemical definitions for the components this input names, from
# files.rcsb.org (~0.6 s). A code that is not fetched raises when folding.
with open('prefetch_ccd.py', 'w') as fh:
  fh.write('import sys, os, importlib.metadata as md\n'
           'from alphafold3.constants import ccd_fetch\n'
           'root = os.path.dirname(md.distribution("alphafold3-colabfold")'
           '.locate_file("alphafold3"))\n'
           'conv = os.path.join(root, "alphafold3", "constants", "converters")\n'
           'os.makedirs(conv, exist_ok=True)\n'
           'ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra=sys.argv[1:]),\n'
           '  os.path.join(conv, "ccd.pickle"),\n'
           '  os.path.join(conv, "chemical_component_sets.pickle"),\n'
           '  libcifpp_dir=os.path.join(root, "share", "libcifpp"))\n')
print(f'Fetching the CCD: 35 standard residues'
      + (f' + {", ".join(ccd_codes)}' if ccd_codes else '') + ' ...')
if os.system('python prefetch_ccd.py ' + ' '.join(ccd_codes)) != 0:
  raise RuntimeError('could not build the CCD tables; see the output above')

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


override: model = 'openbind0'
override: protein = 'SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQ'
override: msa_mode = 'single_sequence'
override: jobname = 'vops'
Fetching the CCD: 35 standard residues ...


Job "vops_a011f"  ->  results will be written to af3_output/vops_a011f/


{'name': 'vops_a011f',
 'sequences': [{'protein': {'id': 'A',
    'sequence': 'SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQ',
    'templates': [],
    'unpairedMsa': '>query\nSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQ\n',
    'pairedMsa': ''}}],
 'modelSeeds': [1],
 'dialect': 'alphafold3',
 'version': 1}

In [ ]:
# 3. The whole fold, volta (attention + tri-mul ops) vs xla, cold then warm.
import os, glob, shutil, subprocess, time
import numpy as np
os.environ['XLA_FLAGS'] = (os.environ.get('XLA_FLAGS', '') +
                           ' --xla_disable_hlo_passes=custom-kernel-fusion-rewriter').strip()

def fold(impl, tag):
  out = f'ab_{tag}'
  shutil.rmtree(out, ignore_errors=True)
  cmd = ['python', 'run_alphafold.py', f'--json_path={json_path}',
         '--model=openbind0', '--norun_data_pipeline', f'--output_dir={out}',
         f'--cache_dir={CACHE_DIR}', f'--lowercache_dir={CACHE_DIR}/lower',
         '--dynamic_recycles', '--force_output_dir',
         f'--flash_attention_implementation={impl}',
         '--num_recycles=3', '--num_diffusion_samples=1', '--num_msa=1024']
  t = time.time()
  p = subprocess.run(cmd, capture_output=True, text=True)
  dt = time.time() - t
  cifs = sorted(glob.glob(f'{out}/**/*.cif', recursive=True))
  if p.returncode or not cifs:
    print(f'--- {tag} FAILED (exit {p.returncode}) ---')
    print(p.stdout[-4000:]); print(p.stderr[-4000:])
    return dt, None
  return dt, cifs[0]

def ca(path):
  xyz = []
  for l in open(path):
    if l.startswith('ATOM'):
      f = l.split()
      if f[3] == 'CA':
        xyz.append([float(f[10]), float(f[11]), float(f[12])])
  return np.array(xyz)

def rmsd(a, b):
  n = min(len(a), len(b))
  a, b = a[:n] - a[:n].mean(0), b[:n] - b[:n].mean(0)
  u, _, vt = np.linalg.svd(a.T @ b)
  d = np.sign(np.linalg.det(u @ vt))
  return float(np.sqrt((((a @ (u @ np.diag([1, 1, d]) @ vt)) - b) ** 2).sum(1).mean()))

res = {}
for impl in ('xla', 'volta'):
  for run in (1, 2):
    dt, cif = fold(impl, f'{impl}{run}')
    print(f'{impl} run {run}: {dt:.1f} s -> {cif}')
    res[(impl, run)] = (dt, cif)
cx, cv = res[('xla', 2)][1], res[('volta', 2)][1]
if cx and cv:
  print(f'warm: xla {res[("xla",2)][0]:.1f} s   volta {res[("volta",2)][0]:.1f} s   '
        f'{res[("xla",2)][0] / res[("volta",2)][0]:.2f}x')
  print(f'CA-RMSD volta vs xla: {rmsd(ca(cx), ca(cv)):.3f} A')
print('RESULT:', 'PASS' if (cx and cv) else 'FAIL')


xla run 1: 130.8 s -> ab_xla1/vops_a011f/seed-1_sample-0/vops_a011f_seed-1_sample-0_model.cif


xla run 2: 81.1 s -> ab_xla2/vops_a011f/seed-1_sample-0/vops_a011f_seed-1_sample-0_model.cif


volta run 1: 104.2 s -> ab_volta1/vops_a011f/seed-1_sample-0/vops_a011f_seed-1_sample-0_model.cif


volta run 2: 70.6 s -> ab_volta2/vops_a011f/seed-1_sample-0/vops_a011f_seed-1_sample-0_model.cif
warm: xla 81.1 s   volta 70.6 s   1.15x
CA-RMSD volta vs xla: 1.092 A
RESULT: PASS
